# Laboratorium 7

Celem siódmego laboratorium jest zapoznanie się oraz zaimplementowanie algorytmu głębokiego uczenia aktywnego - Actor-Critic. Zaimplementowany algorytm będzie testowany z wykorzystaniem środowiska z OpenAI - *CartPole*.


Dołączenie standardowych bibliotek

In [1]:
from collections import deque
import gymnasium as gym
import numpy as np
import random

Dołączenie bibliotek do obsługi sieci neuronowych

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(342)

class Net(nn.Module):
    def __init__(self, state_size, action_size, hidden_neurons, learning_rate, is_softmax=True):
        super(Net, self).__init__()

        self.fc1 = nn.Linear(state_size, hidden_neurons)
        self.fc2 = nn.Linear(hidden_neurons, hidden_neurons)
        self.out = nn.Linear(hidden_neurons, action_size)

        self.is_softmax = is_softmax
        self.learning_rate = learning_rate
        self.optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)
        return F.softmax(x, dim=-1) if self.is_softmax else x
    
    def predict(self, state):
        state = torch.FloatTensor(state)
        with torch.no_grad():
            q_values = self.forward(state)
        
        return q_values.numpy()

## Zadanie 1 - Actor-Critic

<p style='text-align: justify;'>
Celem ćwiczenie jest zaimplementowanie algorytmu Actor-Critic. W tym celu należy utworzyć dwie głębokie sieci neuronowe:
    1. *actor* - sieć, która będzie uczyła się optymalnej strategii (podobna do tej z laboratorium 6),
    2. *critic* - sieć, która będzie uczyła się funkcji oceny stanu (podobnie jak się DQN).
Wagi sieci *actor* aktualizowane są zgodnie ze wzorem:
\begin{equation*}
    \theta \leftarrow \theta + \alpha \delta_t \nabla_\theta log \pi_{\theta}(a_t, s_t | \theta).
\end{equation*}
Wagi sieci *critic* aktualizowane są zgodnie ze wzorem:
\begin{equation*}
    w \leftarrow w + \beta \delta_t \nabla_w\upsilon(s_{t + 1}, w),
\end{equation*}
gdzie:
\begin{equation*}
    \delta_t \leftarrow r_t + \gamma \upsilon(s_{t + 1}, w) - \upsilon(s_t, w).
\end{equation*}
</p>

In [3]:
class Agent:
    def __init__(self, state_size, action_size, actor, critic):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = 0.99    # discount rate
        self.learning_rate = 0.001
        self.actor = actor
        self.critic = critic #critic network should have only one output


    def get_action(self, state):
        """
        Compute the action to take in the current state, basing on policy returned by the network.

        Note: To pick action according to the probability generated by the network
        """

        probabilities = self.actor.predict(state)
        chosen_action = np.random.choice(self.action_size, p=probabilities)
        
        return chosen_action

  

    def learn(self, state, action, reward, next_state, done):
        """
        Function learn networks using information about state, action, reward and next state. 
        First the values for state and next_state should be estimated based on output of critic network.
        Critic network should be trained based on target value:
        target = r + \gamma next_state_value if not done]
        target = r if done.
        Actor network shpuld be trained based on delta value:
        delta = target - state_value
        """
        state_value = self.critic(state)
        next_state_value = self.critic(next_state)

        target = reward + (1 - int(done)) * self.gamma * next_state_value.detach()
        delta = target - state_value

        self.critic.optimizer.zero_grad()
        critic_loss = F.mse_loss(state_value, target)
        critic_loss.backward()
        self.critic.optimizer.step()

        self.actor.optimizer.zero_grad()
        action_probabilities = self.actor(state)
        log_prob = torch.log(action_probabilities[action])
        actor_loss = -log_prob * delta.detach() 
        actor_loss.backward()
        self.actor.optimizer.step() 

        


<>:30: SyntaxWarning: invalid escape sequence '\g'
<>:30: SyntaxWarning: invalid escape sequence '\g'
C:\Users\Filip\AppData\Local\Temp\ipykernel_13960\3070384824.py:30: SyntaxWarning: invalid escape sequence '\g'
  target = r + \gamma next_state_value if not done]


Czas przygotować model sieci, która będzie się uczyła działania w środowisku [*CartPool*](https://gym.openai.com/envs/CartPole-v0/):

In [4]:
env = gym.make("CartPole-v0").env
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
alpha_learning_rate = 0.0001
beta_learning_rate = 0.0005

print(f"State size: {state_size}, Action size: {action_size}")

actor_model =  Net(state_size, action_size, hidden_neurons=128, learning_rate=alpha_learning_rate, is_softmax=True)

critic_model = Net(state_size, 1, hidden_neurons=128, learning_rate=beta_learning_rate, is_softmax=False)

c:\Users\Filip\Documents\mgr-siium\guzw\.venv\Lib\site-packages\gymnasium\envs\registration.py:512: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


State size: 4, Action size: 2


Czas nauczyć agenta gry w środowisku *CartPool*:

In [5]:
agent = Agent(state_size, action_size, actor_model, critic_model)


for i in range(100):
    score_history = []

    for _ in range(100):
        done = False
        score = 0
        state = env.reset()[0]
        state = torch.tensor(state, dtype=torch.float32)
        while not done:
            action = agent.get_action(state)
            next_state, reward, done, _, _ = env.step(action)
            next_state = torch.tensor(next_state, dtype=torch.float32)
            agent.learn(state, action, reward, next_state, done)
            state = next_state
            score += reward
        score_history.append(score)

    print("{} mean reward: {:.3f}".format(i, np.mean(score_history)))

    if np.mean(score_history) > 300:
        print("You Win!")
        break

0 mean reward: 19.580
1 mean reward: 77.640
2 mean reward: 366.560
You Win!
